In [2]:
#######################
# Gravité prédite  (1+2 vs 3+4)  –  Île‑de‑France
#######################
import geopandas as gpd, pandas as pd, folium, torch, numpy as np
from pyrosm import OSM, get_data
from shapely.geometry import Point
from branca.element import Template, MacroElement
from matplotlib import cm
from matplotlib.colors import to_hex
import joblib
import numpy as np
# ── compatibilité pyrosm ≤ 0.6.1 avec NumPy ≥ 1.24 ────────────────
if not hasattr(np, "float"):
    np.float = float

import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, BatchNorm

class GATResNet(nn.Module):
    def __init__(self, in_dim, hidden=32, out_dim=2, heads=8, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.gats = nn.ModuleList([
            GATConv(in_dim, hidden, heads=heads, dropout=dropout),
            GATConv(hidden * heads, hidden, heads=heads, dropout=dropout),
            GATConv(hidden * heads, hidden, heads=heads, dropout=dropout),
        ])
        self.bns = nn.ModuleList([
            BatchNorm(hidden * heads),
            BatchNorm(hidden * heads),
            BatchNorm(hidden * heads),
        ])
        self.res0 = nn.Linear(in_dim, hidden * heads, bias=False)
        self.final = GATConv(hidden * heads, out_dim, heads=1, concat=False)

    def forward(self, data):
        x, ei = data.x, data.edge_index
        h = self.gats[0](x, ei)
        h = F.elu(self.bns[0](h))
        x = self.res0(x) + h
        x = F.dropout(x, p=self.dropout, training=self.training)
        for gat, bn in zip(self.gats[1:], self.bns[1:]):
            h = F.elu(bn(gat(x, ei)))
            x = x + h
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.final(x, ei)


model = joblib.load("../models/GAT.joblib")
data = torch.load("../models/graph_data.pt")
df = pd.read_csv("../data/processed/GAT_df.csv")
shap_values = joblib.load("../models/shap_values.joblib")     
best_thr = joblib.load("../models/best_thr.joblib")

ohe = joblib.load("../models/ohe.joblib")
num_cols = ["vma","TMJA","lat_norm","lon_norm","nb_accidents","dist_hop_m","dist_fire_m"]
cat_cols = ['lit','dep','agg','catr','obs','obsm','situ','prof','plan','infra','circ','int',"bridge","tunnel","traffic_calming"]

feature_names = list(num_cols) + list(ohe.get_feature_names_out(cat_cols))

# ════════════════════════════════════════════════════════════
# 0)  PRÉDICTIONS : logits, probas, seuil
# ════════════════════════════════════════════════════════════
logits   = model(data).softmax(1).cpu()
idx_test = data.test_mask.nonzero(as_tuple=False).view(-1)

probas   = logits[:, 1]                       # p(classe 1) pour tout le graph
preds    = logits.argmax(1)                   # 0 / 1 pour tout le graph

seuil = best_thr                                 # ★ seuil choisi
sel   = (preds[idx_test] == 1) & (probas[idx_test] > seuil)   # ★
chosen_idx = idx_test[sel]                    # indices définitifs
print("✅ accidents graves à projeter :", len(chosen_idx))

# ════════════════════════════════════════════════════════════
# 1)  POINTS  → GeoDataFrame
# ════════════════════════════════════════════════════════════
df_sel = df.loc[chosen_idx.tolist(), ["lat10", "lon10"]].copy()
df_sel["proba"] = probas[chosen_idx].tolist()                 # ★ garder la proba
dec = 4
df_sel["lat"] = df_sel["lat10"] / 10**dec
df_sel["lon"] = df_sel["lon10"] / 10**dec
df_sel["risk"] = 1                                           # toutes = classe 1

gdf_pts = gpd.GeoDataFrame(
    df_sel,
    geometry=gpd.points_from_xy(df_sel.lon, df_sel.lat),
    crs="EPSG:4326"
)

def top3_shap(node_idx: int) -> str:
    vals = shap_values[node_idx]
    idxs = np.argsort(-np.abs(vals))[:3]
    txts = []
    for i in idxs:
        if i < len(feature_names):
            txts.append(f"{feature_names[i]} {vals[i]:+.2f}")
        else:
            txts.append(f"[feat {i}] {vals[i]:+.2f}")
    return " ; ".join(txts)

# 2.  ajouter la colonne aux points sélectionnés -------------------
gdf_pts["shap_top"] = [top3_shap(i) for i in chosen_idx.tolist()]


# ════════════════════════════════════════════════════════════
# 2)  RÉSEAU ROUTIER IDF  (filtré)
# ════════════════════════════════════════════════════════════
ROAD_OK = {
    "motorway","trunk","primary","secondary","tertiary",
    "residential","unclassified","living_street"
}

pbf_path = get_data("ile-de-france")        # adapter pour une autre région
gdf_ways = (
    OSM(pbf_path)
    .get_network("driving")
    .to_crs("EPSG:4326")
)

# 🔽 Supprimer géométries vides ou invalides
gdf_ways = gdf_ways[gdf_ways.geometry.is_valid & ~gdf_ways.geometry.is_empty]

# 🔽 Optionnel : supprimer MultiLineString de longueur 0
gdf_ways = gdf_ways[gdf_ways.geometry.length > 0]

# 🔽 Filtrer par type de route
gdf_ways = (
    gdf_ways
    .loc[lambda x: x["highway"].isin(ROAD_OK), ["geometry"]]
    .reset_index(drop=True)
)

print(f"✅ réseau filtré : {len(gdf_ways):,} tronçons")

# ════════════════════════════════════════════════════════════
# 3)  JOINTURE « plus proche »  (≤ 20 m)
# ════════════════════════════════════════════════════════════
gdf_join = gpd.sjoin_nearest(
    gdf_ways,
    gdf_pts[["risk", "proba", "shap_top", "geometry"]],      # ★ on joint aussi la proba
    how="left",
    distance_col="d",
    max_distance=20/111_000      # 20 m
)

# harmoniser les noms éventuels (_right)
for col in ["risk", "proba", "shap_top"]:
    if col not in gdf_join.columns and f"{col}_right" in gdf_join.columns:
        gdf_join = gdf_join.rename(columns={f"{col}_right": col})

gdf_join["risk"]  = gdf_join["risk"].fillna(0).astype(int)
gdf_join["proba"] = gdf_join["proba"].fillna(0)              # 0 si aucun point
gdf_join["shap_top"] = gdf_join["shap_top"].fillna("—")  # tiret si aucun point


# ════════════════════════════════════════════════════════════
# 4)  mini‑lissage optionnel (tronçon + 2 voisins < 200 m)
# ════════════════════════════════════════════════════════════
gdf_risk = gdf_join.query("risk == 1")[["proba", "geometry"]]
if len(gdf_risk) >= 10:
    gdf_risk = gdf_risk.reset_index(drop=False)
    neigh = (
        gpd.sjoin_nearest(
            gdf_risk, gdf_risk,
            how="left", distance_col="dist",
            max_distance=0.002, rsuffix="n"       # 200 m
        )
        .sort_values(["index_left", "dist"])
        .groupby("index_left").head(3)
        .groupby("index_left")["proba_n"].mean()
    )
    gdf_join.loc[neigh.index, "proba"] = neigh   # ★ on lisse la proba
    gdf_join.loc[neigh.index, "risk"]  = 1

print("✅ distribution finale :\n", gdf_join["risk"].value_counts())

# ════════════════════════════════════════════════════════════
# 5)  CARTE FOLIUM
# ════════════════════════════════════════════════════════════


# Palette rouge graduée
delta  = max(1e-6, 1.0 - best_thr)
reds   = [to_hex(c) for c in cm.Reds(np.linspace(0.4, 1, 9))]
labels = [f"{best_thr + i * delta / 8:.2f}" for i in range(9)]

def style_fun(feat):
    p = feat["properties"]["proba"]
    idx = int(min(max((p - best_thr) / delta, 0), 1) * 8)
    return {"color": reds[idx], "weight": 2, "opacity": 0.8}

m = folium.Map(location=[48.8, 2.35], zoom_start=9, tiles="CartoDB positron")

g_high = gdf_join[gdf_join["risk"] == 1].copy()
g_high["geometry"] = g_high.geometry.simplify(0.00005)

tooltip = folium.features.GeoJsonTooltip(
    fields=["proba", "shap_top"],
    aliases=["Proba :", "Top SHAP :"],
    localize=True
)

folium.GeoJson(g_high, style_function=style_fun, tooltip=tooltip).add_to(m)

# --- Légende colorbar -------------------------------------------------
rows = "\n".join(
    f'<rect x="0" y="{i*15}" width="20" height="15" style="fill:{reds[i]};" />'
    f'<text x="25" y="{i*15 + 12}" font-size="12">{labels[i]}</text>'
    for i in range(9)
)
legend_html = f"""
{{% macro html(this, kwargs) %}}
<div style="
    position: fixed;
    bottom: 50px;
    left: 50px;
    width: 180px;
    z-index:9999;
    font-size:14px;
    background-color: white;
    padding: 10px;
    border:2px solid grey;
    border-radius:5px;
    box-shadow:2px 2px 6px rgba(0,0,0,0.3);
">
<b>Probabilité (gravité)</b><br>
<svg width="150" height="140">
{rows}
</svg>
</div>
{{% endmacro %}}
"""
legend = MacroElement()
legend._template = Template(legend_html)
m.get_root().add_child(legend)

folium.LayerControl().add_to(m)
m.save("../reports/figures/pred_grav_idf.html")   # export HTML
m # affiche la map

/var/folders/05/_vxzr9653lg058wd1tp1rdyr0000gn/T/ipykernel_1467/3592510002.py:52: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/GAT_df.csv")


✅ accidents graves à projeter : 3403


/opt/anaconda3/envs/env_torch/lib/python3.10/site-packages/pyrosm/networks.py:37: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  edges, nodes = prepare_geodataframe(
/var/folders/05/_vxzr9653lg058wd1tp1rdyr0000gn/T/ipykernel_1467/3592510002.p

✅ réseau filtré : 262,633 tronçons


/opt/anaconda3/envs/env_torch/lib/python3.10/site-packages/geopandas/array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


✅ distribution finale :
 risk
0    262493
1       140
Name: count, dtype: int64


/opt/anaconda3/envs/env_torch/lib/python3.10/site-packages/geopandas/array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


In [3]:
#######################
# Gravité prédite  (1+2 vs 3+4)  –  Île‑de‑France
#######################
import geopandas as gpd, pandas as pd, folium, torch, numpy as np
from pyrosm import OSM, get_data
from shapely.geometry import Point
from branca.element import Template, MacroElement
from matplotlib import cm
from matplotlib.colors import to_hex
import joblib
import numpy as np
# ── compatibilité pyrosm ≤ 0.6.1 avec NumPy ≥ 1.24 ────────────────
if not hasattr(np, "float"):
    np.float = float

import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, BatchNorm

class GATResNet(nn.Module):
    def __init__(self, in_dim, hidden=32, out_dim=2, heads=8, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.gats = nn.ModuleList([
            GATConv(in_dim, hidden, heads=heads, dropout=dropout),
            GATConv(hidden * heads, hidden, heads=heads, dropout=dropout),
            GATConv(hidden * heads, hidden, heads=heads, dropout=dropout),
        ])
        self.bns = nn.ModuleList([
            BatchNorm(hidden * heads),
            BatchNorm(hidden * heads),
            BatchNorm(hidden * heads),
        ])
        self.res0 = nn.Linear(in_dim, hidden * heads, bias=False)
        self.final = GATConv(hidden * heads, out_dim, heads=1, concat=False)

    def forward(self, data):
        x, ei = data.x, data.edge_index
        h = self.gats[0](x, ei)
        h = F.elu(self.bns[0](h))
        x = self.res0(x) + h
        x = F.dropout(x, p=self.dropout, training=self.training)
        for gat, bn in zip(self.gats[1:], self.bns[1:]):
            h = F.elu(bn(gat(x, ei)))
            x = x + h
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.final(x, ei)


model = joblib.load("../models/GAT.joblib")
data = torch.load("../models/graph_data.pt")
df = pd.read_csv("../data/processed/GAT_df.csv")
shap_values = joblib.load("../models/shap_values.joblib")     
best_thr = joblib.load("../models/best_thr.joblib")

ohe = joblib.load("../models/ohe.joblib")
num_cols = ["vma","TMJA","lat_norm","lon_norm","nb_accidents","dist_hop_m","dist_fire_m"]
cat_cols = ['lit','dep','agg','catr','obs','obsm','situ','prof','plan','infra','circ','int',"bridge","tunnel","traffic_calming"]

feature_names = list(num_cols) + list(ohe.get_feature_names_out(cat_cols))

# ════════════════════════════════════════════════════════════
# 0)  PRÉDICTIONS : logits, probas, seuil
# ════════════════════════════════════════════════════════════
logits   = model(data).softmax(1).cpu()
idx_test = data.test_mask.nonzero(as_tuple=False).view(-1)

probas   = logits[:, 1]                       # p(classe 1) pour tout le graph
preds    = logits.argmax(1)                   # 0 / 1 pour tout le graph

seuil = best_thr                                 # ★ seuil choisi
sel   = (preds[idx_test] == 1) & (probas[idx_test] > seuil)   # ★
chosen_idx = idx_test[sel]                    # indices définitifs
print("✅ accidents graves à projeter :", len(chosen_idx))

# ════════════════════════════════════════════════════════════
# 1)  POINTS  → GeoDataFrame
# ════════════════════════════════════════════════════════════
df_sel = df.loc[chosen_idx.tolist(), ["lat10", "lon10"]].copy()
df_sel["proba"] = probas[chosen_idx].tolist()                 # ★ garder la proba
dec = 4
df_sel["lat"] = df_sel["lat10"] / 10**dec
df_sel["lon"] = df_sel["lon10"] / 10**dec
df_sel["risk"] = 1                                           # toutes = classe 1

gdf_pts = gpd.GeoDataFrame(
    df_sel,
    geometry=gpd.points_from_xy(df_sel.lon, df_sel.lat),
    crs="EPSG:4326"
)

def top3_shap(node_idx: int) -> str:
    vals = shap_values[node_idx]
    idxs = np.argsort(-np.abs(vals))[:3]
    txts = []
    for i in idxs:
        if i < len(feature_names):
            txts.append(f"{feature_names[i]} {vals[i]:+.2f}")
        else:
            txts.append(f"[feat {i}] {vals[i]:+.2f}")
    return " ; ".join(txts)

# 2.  ajouter la colonne aux points sélectionnés -------------------
gdf_pts["shap_top"] = [top3_shap(i) for i in chosen_idx.tolist()]


# ════════════════════════════════════════════════════════════
# 2)  RÉSEAU ROUTIER IDF  (filtré)
# ════════════════════════════════════════════════════════════
ROAD_OK = {
    "motorway","trunk","primary","secondary","tertiary",
    "residential","unclassified","living_street"
}

pbf_path = get_data("provence-alpes-cote-d-azur")        # adapter pour une autre région
gdf_ways = (
    OSM(pbf_path)
    .get_network("driving")
    .to_crs("EPSG:4326")
)

# 🔽 Supprimer géométries vides ou invalides
gdf_ways = gdf_ways[gdf_ways.geometry.is_valid & ~gdf_ways.geometry.is_empty]

# 🔽 Optionnel : supprimer MultiLineString de longueur 0
gdf_ways = gdf_ways[gdf_ways.geometry.length > 0]

# 🔽 Filtrer par type de route
gdf_ways = (
    gdf_ways
    .loc[lambda x: x["highway"].isin(ROAD_OK), ["geometry"]]
    .reset_index(drop=True)
)

print(f"✅ réseau filtré : {len(gdf_ways):,} tronçons")

# ════════════════════════════════════════════════════════════
# 3)  JOINTURE « plus proche »  (≤ 20 m)
# ════════════════════════════════════════════════════════════
gdf_join = gpd.sjoin_nearest(
    gdf_ways,
    gdf_pts[["risk", "proba", "shap_top", "geometry"]],      # ★ on joint aussi la proba
    how="left",
    distance_col="d",
    max_distance=20/111_000      # 20 m
)

# harmoniser les noms éventuels (_right)
for col in ["risk", "proba", "shap_top"]:
    if col not in gdf_join.columns and f"{col}_right" in gdf_join.columns:
        gdf_join = gdf_join.rename(columns={f"{col}_right": col})

gdf_join["risk"]  = gdf_join["risk"].fillna(0).astype(int)
gdf_join["proba"] = gdf_join["proba"].fillna(0)              # 0 si aucun point
gdf_join["shap_top"] = gdf_join["shap_top"].fillna("—")  # tiret si aucun point


# ════════════════════════════════════════════════════════════
# 4)  mini‑lissage optionnel (tronçon + 2 voisins < 200 m)
# ════════════════════════════════════════════════════════════
gdf_risk = gdf_join.query("risk == 1")[["proba", "geometry"]]
if len(gdf_risk) >= 10:
    gdf_risk = gdf_risk.reset_index(drop=False)
    neigh = (
        gpd.sjoin_nearest(
            gdf_risk, gdf_risk,
            how="left", distance_col="dist",
            max_distance=0.002, rsuffix="n"       # 200 m
        )
        .sort_values(["index_left", "dist"])
        .groupby("index_left").head(3)
        .groupby("index_left")["proba_n"].mean()
    )
    gdf_join.loc[neigh.index, "proba"] = neigh   # ★ on lisse la proba
    gdf_join.loc[neigh.index, "risk"]  = 1

print("✅ distribution finale :\n", gdf_join["risk"].value_counts())

# ════════════════════════════════════════════════════════════
# 5)  CARTE FOLIUM
# ════════════════════════════════════════════════════════════


# Palette rouge graduée
delta  = max(1e-6, 1.0 - best_thr)
reds   = [to_hex(c) for c in cm.Reds(np.linspace(0.4, 1, 9))]
labels = [f"{best_thr + i * delta / 8:.2f}" for i in range(9)]

def style_fun(feat):
    p = feat["properties"]["proba"]
    idx = int(min(max((p - best_thr) / delta, 0), 1) * 8)
    return {"color": reds[idx], "weight": 2, "opacity": 0.8}

m = folium.Map(location=[43.8, 6.2], zoom_start=8, tiles="CartoDB positron")  # A adapter à la région

g_high = gdf_join[gdf_join["risk"] == 1].copy()
g_high["geometry"] = g_high.geometry.simplify(0.00005)

tooltip = folium.features.GeoJsonTooltip(
    fields=["proba", "shap_top"],
    aliases=["Proba :", "Top SHAP :"],
    localize=True
)

folium.GeoJson(g_high, style_function=style_fun, tooltip=tooltip).add_to(m)

# --- Légende colorbar -------------------------------------------------
rows = "\n".join(
    f'<rect x="0" y="{i*15}" width="20" height="15" style="fill:{reds[i]};" />'
    f'<text x="25" y="{i*15 + 12}" font-size="12">{labels[i]}</text>'
    for i in range(9)
)
legend_html = f"""
{{% macro html(this, kwargs) %}}
<div style="
    position: fixed;
    bottom: 50px;
    left: 50px;
    width: 180px;
    z-index:9999;
    font-size:14px;
    background-color: white;
    padding: 10px;
    border:2px solid grey;
    border-radius:5px;
    box-shadow:2px 2px 6px rgba(0,0,0,0.3);
">
<b>Probabilité (gravité)</b><br>
<svg width="150" height="140">
{rows}
</svg>
</div>
{{% endmacro %}}
"""
legend = MacroElement()
legend._template = Template(legend_html)
m.get_root().add_child(legend)

folium.LayerControl().add_to(m)
m.save("../reports/figures/pred_grav_PACA.html")   # export HTML A adapter à la région
m # affiche la map

/var/folders/05/_vxzr9653lg058wd1tp1rdyr0000gn/T/ipykernel_1467/2194646152.py:52: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/GAT_df.csv")


✅ accidents graves à projeter : 3403
Downloaded Protobuf data 'provence-alpes-cote-d-azur-latest.osm.pbf' (351.27 MB) to:
'/var/folders/05/_vxzr9653lg058wd1tp1rdyr0000gn/T/pyrosm/provence-alpes-cote-d-azur-latest.osm.pbf'


/opt/anaconda3/envs/env_torch/lib/python3.10/site-packages/pyrosm/networks.py:37: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  edges, nodes = prepare_geodataframe(
/var/folders/05/_vxzr9653lg058wd1tp1rdyr0000gn/T/ipykernel_1467/2194646152.p

✅ réseau filtré : 260,305 tronçons


/opt/anaconda3/envs/env_torch/lib/python3.10/site-packages/geopandas/array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
/opt/anaconda3/envs/env_torch/lib/python3.10/site-packages/geopandas/array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


✅ distribution finale :
 risk
0    259848
1       457
Name: count, dtype: int64


In [4]:
#######################
# Gravité prédite  (1+2 vs 3+4)  –  Bretagne
#######################
import geopandas as gpd, pandas as pd, folium, torch, numpy as np
from pyrosm import OSM, get_data
from shapely.geometry import Point
from branca.element import Template, MacroElement
from matplotlib import cm
from matplotlib.colors import to_hex
import joblib
import numpy as np
# ── compatibilité pyrosm ≤ 0.6.1 avec NumPy ≥ 1.24 ────────────────
if not hasattr(np, "float"):
    np.float = float

import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, BatchNorm

class GATResNet(nn.Module):
    def __init__(self, in_dim, hidden=32, out_dim=2, heads=8, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.gats = nn.ModuleList([
            GATConv(in_dim, hidden, heads=heads, dropout=dropout),
            GATConv(hidden * heads, hidden, heads=heads, dropout=dropout),
            GATConv(hidden * heads, hidden, heads=heads, dropout=dropout),
        ])
        self.bns = nn.ModuleList([
            BatchNorm(hidden * heads),
            BatchNorm(hidden * heads),
            BatchNorm(hidden * heads),
        ])
        self.res0 = nn.Linear(in_dim, hidden * heads, bias=False)
        self.final = GATConv(hidden * heads, out_dim, heads=1, concat=False)

    def forward(self, data):
        x, ei = data.x, data.edge_index
        h = self.gats[0](x, ei)
        h = F.elu(self.bns[0](h))
        x = self.res0(x) + h
        x = F.dropout(x, p=self.dropout, training=self.training)
        for gat, bn in zip(self.gats[1:], self.bns[1:]):
            h = F.elu(bn(gat(x, ei)))
            x = x + h
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.final(x, ei)


model = joblib.load("../models/GAT.joblib")
data = torch.load("../models/graph_data.pt")
df = pd.read_csv("../data/processed/GAT_df.csv")
shap_values = joblib.load("../models/shap_values.joblib")     
best_thr = joblib.load("../models/best_thr.joblib")

ohe = joblib.load("../models/ohe.joblib")
num_cols = ["vma","TMJA","lat_norm","lon_norm","nb_accidents","dist_hop_m","dist_fire_m"]
cat_cols = ['lit','dep','agg','catr','obs','obsm','situ','prof','plan','infra','circ','int',"bridge","tunnel","traffic_calming"]

feature_names = list(num_cols) + list(ohe.get_feature_names_out(cat_cols))

# ════════════════════════════════════════════════════════════
# 0)  PRÉDICTIONS : logits, probas, seuil
# ════════════════════════════════════════════════════════════
logits   = model(data).softmax(1).cpu()
idx_test = data.test_mask.nonzero(as_tuple=False).view(-1)

probas   = logits[:, 1]                       # p(classe 1) pour tout le graph
preds    = logits.argmax(1)                   # 0 / 1 pour tout le graph

seuil = best_thr                                 # ★ seuil choisi
sel   = (preds[idx_test] == 1) & (probas[idx_test] > seuil)   # ★
chosen_idx = idx_test[sel]                    # indices définitifs
print("✅ accidents graves à projeter :", len(chosen_idx))

# ════════════════════════════════════════════════════════════
# 1)  POINTS  → GeoDataFrame
# ════════════════════════════════════════════════════════════
df_sel = df.loc[chosen_idx.tolist(), ["lat10", "lon10"]].copy()
df_sel["proba"] = probas[chosen_idx].tolist()                 # ★ garder la proba
dec = 4
df_sel["lat"] = df_sel["lat10"] / 10**dec
df_sel["lon"] = df_sel["lon10"] / 10**dec
df_sel["risk"] = 1                                           # toutes = classe 1

gdf_pts = gpd.GeoDataFrame(
    df_sel,
    geometry=gpd.points_from_xy(df_sel.lon, df_sel.lat),
    crs="EPSG:4326"
)

def top3_shap(node_idx: int) -> str:
    vals = shap_values[node_idx]
    idxs = np.argsort(-np.abs(vals))[:3]
    txts = []
    for i in idxs:
        if i < len(feature_names):
            txts.append(f"{feature_names[i]} {vals[i]:+.2f}")
        else:
            txts.append(f"[feat {i}] {vals[i]:+.2f}")
    return " ; ".join(txts)

# 2.  ajouter la colonne aux points sélectionnés -------------------
gdf_pts["shap_top"] = [top3_shap(i) for i in chosen_idx.tolist()]


# ════════════════════════════════════════════════════════════
# 2)  RÉSEAU ROUTIER IDF  (filtré)
# ════════════════════════════════════════════════════════════
ROAD_OK = {
    "motorway","trunk","primary","secondary","tertiary",
    "residential","unclassified","living_street"
}

pbf_path = get_data("bretagne")        # adapter pour une autre région
gdf_ways = (
    OSM(pbf_path)
    .get_network("driving")
    .to_crs("EPSG:4326")
)

# 🔽 Supprimer géométries vides ou invalides
gdf_ways = gdf_ways[gdf_ways.geometry.is_valid & ~gdf_ways.geometry.is_empty]

# 🔽 Optionnel : supprimer MultiLineString de longueur 0
gdf_ways = gdf_ways[gdf_ways.geometry.length > 0]

# 🔽 Filtrer par type de route
gdf_ways = (
    gdf_ways
    .loc[lambda x: x["highway"].isin(ROAD_OK), ["geometry"]]
    .reset_index(drop=True)
)

print(f"✅ réseau filtré : {len(gdf_ways):,} tronçons")

# ════════════════════════════════════════════════════════════
# 3)  JOINTURE « plus proche »  (≤ 20 m)
# ════════════════════════════════════════════════════════════
gdf_join = gpd.sjoin_nearest(
    gdf_ways,
    gdf_pts[["risk", "proba", "shap_top", "geometry"]],      # ★ on joint aussi la proba
    how="left",
    distance_col="d",
    max_distance=20/111_000      # 20 m
)

# harmoniser les noms éventuels (_right)
for col in ["risk", "proba", "shap_top"]:
    if col not in gdf_join.columns and f"{col}_right" in gdf_join.columns:
        gdf_join = gdf_join.rename(columns={f"{col}_right": col})

gdf_join["risk"]  = gdf_join["risk"].fillna(0).astype(int)
gdf_join["proba"] = gdf_join["proba"].fillna(0)              # 0 si aucun point
gdf_join["shap_top"] = gdf_join["shap_top"].fillna("—")  # tiret si aucun point


# ════════════════════════════════════════════════════════════
# 4)  mini‑lissage optionnel (tronçon + 2 voisins < 200 m)
# ════════════════════════════════════════════════════════════
gdf_risk = gdf_join.query("risk == 1")[["proba", "geometry"]]
if len(gdf_risk) >= 10:
    gdf_risk = gdf_risk.reset_index(drop=False)
    neigh = (
        gpd.sjoin_nearest(
            gdf_risk, gdf_risk,
            how="left", distance_col="dist",
            max_distance=0.002, rsuffix="n"       # 200 m
        )
        .sort_values(["index_left", "dist"])
        .groupby("index_left").head(3)
        .groupby("index_left")["proba_n"].mean()
    )
    gdf_join.loc[neigh.index, "proba"] = neigh   # ★ on lisse la proba
    gdf_join.loc[neigh.index, "risk"]  = 1

print("✅ distribution finale :\n", gdf_join["risk"].value_counts())

# ════════════════════════════════════════════════════════════
# 5)  CARTE FOLIUM
# ════════════════════════════════════════════════════════════


# Palette rouge graduée
delta  = max(1e-6, 1.0 - best_thr)
reds   = [to_hex(c) for c in cm.Reds(np.linspace(0.4, 1, 9))]
labels = [f"{best_thr + i * delta / 8:.2f}" for i in range(9)]

def style_fun(feat):
    p = feat["properties"]["proba"]
    idx = int(min(max((p - best_thr) / delta, 0), 1) * 8)
    return {"color": reds[idx], "weight": 2, "opacity": 0.8}

m = folium.Map(location=[48, -3], zoom_start=8, tiles="CartoDB positron")  # A adapter pour la région

g_high = gdf_join[gdf_join["risk"] == 1].copy()
g_high["geometry"] = g_high.geometry.simplify(0.00005)

tooltip = folium.features.GeoJsonTooltip(
    fields=["proba", "shap_top"],
    aliases=["Proba :", "Top SHAP :"],
    localize=True
)

folium.GeoJson(g_high, style_function=style_fun, tooltip=tooltip).add_to(m)

# --- Légende colorbar -------------------------------------------------
rows = "\n".join(
    f'<rect x="0" y="{i*15}" width="20" height="15" style="fill:{reds[i]};" />'
    f'<text x="25" y="{i*15 + 12}" font-size="12">{labels[i]}</text>'
    for i in range(9)
)
legend_html = f"""
{{% macro html(this, kwargs) %}}
<div style="
    position: fixed;
    bottom: 50px;
    left: 50px;
    width: 180px;
    z-index:9999;
    font-size:14px;
    background-color: white;
    padding: 10px;
    border:2px solid grey;
    border-radius:5px;
    box-shadow:2px 2px 6px rgba(0,0,0,0.3);
">
<b>Probabilité (gravité)</b><br>
<svg width="150" height="140">
{rows}
</svg>
</div>
{{% endmacro %}}
"""
legend = MacroElement()
legend._template = Template(legend_html)
m.get_root().add_child(legend)

folium.LayerControl().add_to(m)
m.save("../reports/figures/pred_grav_bretagne.html")   # export HTML
m # affiche la map

/var/folders/05/_vxzr9653lg058wd1tp1rdyr0000gn/T/ipykernel_1467/2696547564.py:52: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/GAT_df.csv")


✅ accidents graves à projeter : 3403
Downloaded Protobuf data 'bretagne-latest.osm.pbf' (298.38 MB) to:
'/var/folders/05/_vxzr9653lg058wd1tp1rdyr0000gn/T/pyrosm/bretagne-latest.osm.pbf'


/opt/anaconda3/envs/env_torch/lib/python3.10/site-packages/pyrosm/networks.py:37: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  edges, nodes = prepare_geodataframe(
/var/folders/05/_vxzr9653lg058wd1tp1rdyr0000gn/T/ipykernel_1467/2696547564.p

✅ réseau filtré : 316,993 tronçons


/opt/anaconda3/envs/env_torch/lib/python3.10/site-packages/geopandas/array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
/opt/anaconda3/envs/env_torch/lib/python3.10/site-packages/geopandas/array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


✅ distribution finale :
 risk
0    316707
1       286
Name: count, dtype: int64


In [7]:
#######################
# Gravité prédite  (1+2 vs 3+4)  –  Corse
#######################
import geopandas as gpd, pandas as pd, folium, torch, numpy as np
from pyrosm import OSM, get_data
from shapely.geometry import Point
from branca.element import Template, MacroElement
from matplotlib import cm
from matplotlib.colors import to_hex
import joblib
import numpy as np
# ── compatibilité pyrosm ≤ 0.6.1 avec NumPy ≥ 1.24 ────────────────
if not hasattr(np, "float"):
    np.float = float

import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, BatchNorm

class GATResNet(nn.Module):
    def __init__(self, in_dim, hidden=32, out_dim=2, heads=8, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.gats = nn.ModuleList([
            GATConv(in_dim, hidden, heads=heads, dropout=dropout),
            GATConv(hidden * heads, hidden, heads=heads, dropout=dropout),
            GATConv(hidden * heads, hidden, heads=heads, dropout=dropout),
        ])
        self.bns = nn.ModuleList([
            BatchNorm(hidden * heads),
            BatchNorm(hidden * heads),
            BatchNorm(hidden * heads),
        ])
        self.res0 = nn.Linear(in_dim, hidden * heads, bias=False)
        self.final = GATConv(hidden * heads, out_dim, heads=1, concat=False)

    def forward(self, data):
        x, ei = data.x, data.edge_index
        h = self.gats[0](x, ei)
        h = F.elu(self.bns[0](h))
        x = self.res0(x) + h
        x = F.dropout(x, p=self.dropout, training=self.training)
        for gat, bn in zip(self.gats[1:], self.bns[1:]):
            h = F.elu(bn(gat(x, ei)))
            x = x + h
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.final(x, ei)


model = joblib.load("../models/GAT.joblib")
data = torch.load("../models/graph_data.pt")
df = pd.read_csv("../data/processed/GAT_df.csv")
shap_values = joblib.load("../models/shap_values.joblib")     
best_thr = joblib.load("../models/best_thr.joblib")

ohe = joblib.load("../models/ohe.joblib")
num_cols = ["vma","TMJA","lat_norm","lon_norm","nb_accidents","dist_hop_m","dist_fire_m"]
cat_cols = ['lit','dep','agg','catr','obs','obsm','situ','prof','plan','infra','circ','int',"bridge","tunnel","traffic_calming"]

feature_names = list(num_cols) + list(ohe.get_feature_names_out(cat_cols))

# ════════════════════════════════════════════════════════════
# 0)  PRÉDICTIONS : logits, probas, seuil
# ════════════════════════════════════════════════════════════
logits   = model(data).softmax(1).cpu()
idx_test = data.test_mask.nonzero(as_tuple=False).view(-1)

probas   = logits[:, 1]                       # p(classe 1) pour tout le graph
preds    = logits.argmax(1)                   # 0 / 1 pour tout le graph

seuil = best_thr                                 # ★ seuil choisi
sel   = (preds[idx_test] == 1) & (probas[idx_test] > seuil)   # ★
chosen_idx = idx_test[sel]                    # indices définitifs
print("✅ accidents graves à projeter :", len(chosen_idx))

# ════════════════════════════════════════════════════════════
# 1)  POINTS  → GeoDataFrame
# ════════════════════════════════════════════════════════════
df_sel = df.loc[chosen_idx.tolist(), ["lat10", "lon10"]].copy()
df_sel["proba"] = probas[chosen_idx].tolist()                 # ★ garder la proba
dec = 4
df_sel["lat"] = df_sel["lat10"] / 10**dec
df_sel["lon"] = df_sel["lon10"] / 10**dec
df_sel["risk"] = 1                                           # toutes = classe 1

gdf_pts = gpd.GeoDataFrame(
    df_sel,
    geometry=gpd.points_from_xy(df_sel.lon, df_sel.lat),
    crs="EPSG:4326"
)

def top3_shap(node_idx: int) -> str:
    vals = shap_values[node_idx]
    idxs = np.argsort(-np.abs(vals))[:3]
    txts = []
    for i in idxs:
        if i < len(feature_names):
            txts.append(f"{feature_names[i]} {vals[i]:+.2f}")
        else:
            txts.append(f"[feat {i}] {vals[i]:+.2f}")
    return " ; ".join(txts)

# 2.  ajouter la colonne aux points sélectionnés -------------------
gdf_pts["shap_top"] = [top3_shap(i) for i in chosen_idx.tolist()]


# ════════════════════════════════════════════════════════════
# 2)  RÉSEAU ROUTIER   (filtré)
# ════════════════════════════════════════════════════════════
ROAD_OK = {
    "motorway","trunk","primary","secondary","tertiary",
    "residential","unclassified","living_street"
}

pbf_path = get_data("corse")        # adapter pour une autre région
gdf_ways = (
    OSM(pbf_path)
    .get_network("driving")
    .to_crs("EPSG:4326")
)

# 🔽 Supprimer géométries vides ou invalides
gdf_ways = gdf_ways[gdf_ways.geometry.is_valid & ~gdf_ways.geometry.is_empty]

# 🔽 Optionnel : supprimer MultiLineString de longueur 0
gdf_ways = gdf_ways[gdf_ways.geometry.length > 0]

# 🔽 Filtrer par type de route
gdf_ways = (
    gdf_ways
    .loc[lambda x: x["highway"].isin(ROAD_OK), ["geometry"]]
    .reset_index(drop=True)
)

print(f"✅ réseau filtré : {len(gdf_ways):,} tronçons")

# ════════════════════════════════════════════════════════════
# 3)  JOINTURE « plus proche »  (≤ 20 m)
# ════════════════════════════════════════════════════════════
gdf_join = gpd.sjoin_nearest(
    gdf_ways,
    gdf_pts[["risk", "proba", "shap_top", "geometry"]],      # ★ on joint aussi la proba
    how="left",
    distance_col="d",
    max_distance=20/111_000      # 20 m
)

# harmoniser les noms éventuels (_right)
for col in ["risk", "proba", "shap_top"]:
    if col not in gdf_join.columns and f"{col}_right" in gdf_join.columns:
        gdf_join = gdf_join.rename(columns={f"{col}_right": col})

gdf_join["risk"]  = gdf_join["risk"].fillna(0).astype(int)
gdf_join["proba"] = gdf_join["proba"].fillna(0)              # 0 si aucun point
gdf_join["shap_top"] = gdf_join["shap_top"].fillna("—")  # tiret si aucun point


# ════════════════════════════════════════════════════════════
# 4)  mini‑lissage optionnel (tronçon + 2 voisins < 200 m)
# ════════════════════════════════════════════════════════════
gdf_risk = gdf_join.query("risk == 1")[["proba", "geometry"]]
if len(gdf_risk) >= 10:
    gdf_risk = gdf_risk.reset_index(drop=False)
    neigh = (
        gpd.sjoin_nearest(
            gdf_risk, gdf_risk,
            how="left", distance_col="dist",
            max_distance=0.002, rsuffix="n"       # 200 m
        )
        .sort_values(["index_left", "dist"])
        .groupby("index_left").head(3)
        .groupby("index_left")["proba_n"].mean()
    )
    gdf_join.loc[neigh.index, "proba"] = neigh   # ★ on lisse la proba
    gdf_join.loc[neigh.index, "risk"]  = 1

print("✅ distribution finale :\n", gdf_join["risk"].value_counts())

# ════════════════════════════════════════════════════════════
# 5)  CARTE FOLIUM
# ════════════════════════════════════════════════════════════


# Palette rouge graduée
delta  = max(1e-6, 1.0 - best_thr)
reds   = [to_hex(c) for c in cm.Reds(np.linspace(0.4, 1, 9))]
labels = [f"{best_thr + i * delta / 8:.2f}" for i in range(9)]

def style_fun(feat):
    p = feat["properties"]["proba"]
    idx = int(min(max((p - best_thr) / delta, 0), 1) * 8)
    return {"color": reds[idx], "weight": 2, "opacity": 0.8}

m = folium.Map(location=[42.3, 9], zoom_start=8, tiles="CartoDB positron")  # A adapter pour la région

g_high = gdf_join[gdf_join["risk"] == 1].copy()
g_high["geometry"] = g_high.geometry.simplify(0.00005)

tooltip = folium.features.GeoJsonTooltip(
    fields=["proba", "shap_top"],
    aliases=["Proba :", "Top SHAP :"],
    localize=True
)

folium.GeoJson(g_high, style_function=style_fun, tooltip=tooltip).add_to(m)

# --- Légende colorbar -------------------------------------------------
rows = "\n".join(
    f'<rect x="0" y="{i*15}" width="20" height="15" style="fill:{reds[i]};" />'
    f'<text x="25" y="{i*15 + 12}" font-size="12">{labels[i]}</text>'
    for i in range(9)
)
legend_html = f"""
{{% macro html(this, kwargs) %}}
<div style="
    position: fixed;
    bottom: 50px;
    left: 50px;
    width: 180px;
    z-index:9999;
    font-size:14px;
    background-color: white;
    padding: 10px;
    border:2px solid grey;
    border-radius:5px;
    box-shadow:2px 2px 6px rgba(0,0,0,0.3);
">
<b>Probabilité (gravité)</b><br>
<svg width="150" height="140">
{rows}
</svg>
</div>
{{% endmacro %}}
"""
legend = MacroElement()
legend._template = Template(legend_html)
m.get_root().add_child(legend)

folium.LayerControl().add_to(m)
m.save("../reports/figures/pred_grav_corse.html")   # export HTML
m # affiche la map

/var/folders/05/_vxzr9653lg058wd1tp1rdyr0000gn/T/ipykernel_9396/3646307426.py:52: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/GAT_df.csv")


✅ accidents graves à projeter : 3175


/opt/anaconda3/envs/env_torch/lib/python3.10/site-packages/pyrosm/networks.py:37: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  edges, nodes = prepare_geodataframe(


✅ réseau filtré : 20,347 tronçons
✅ distribution finale :
 risk
0    20330
1       17
Name: count, dtype: int64


/var/folders/05/_vxzr9653lg058wd1tp1rdyr0000gn/T/ipykernel_9396/3646307426.py:126: UserWarning: Geometry is in a geographic CRS. Results from 'length' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_ways = gdf_ways[gdf_ways.geometry.length > 0]
/opt/anaconda3/envs/env_torch/lib/python3.10/site-packages/geopandas/array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
/opt/anaconda3/envs/env_torch/lib/python3.10/site-packages/geopandas/array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


In [8]:
"""
Titre : Carte France entière des tronçons à gravité élevée
Date   : 2025-07-08

Description
-----------
Ce script charge le modèle GAT entraîné, calcule les prédictions de gravité
pour l’ensemble du jeu de données puis, région par région (anciens découpages
Geofabrik), projette ces risques sur le réseau routier OpenStreetMap.
Le résultat est exporté sous la forme d’une carte Folium
`pred_grav_france.html`.

Compatibilité
-------------
- Python >= 3.10
- numpy >= 1.24           (patch compatibilité pyrosm <=0.6.1 inclus)
- geopandas >= 0.13
- shapely >= 2.0
- pyrosm <= 0.6.1
- folium >= 0.15
- torch >= 2.1
- torch_geometric >= 2.4
- branca, matplotlib, joblib

Installation rapide :
    pip install geopandas folium pyrosm torch torch_geometric shapely branca joblib matplotlib

Répertoires attendus :
    ../models/          (GAT.joblib, graph_data.pt, shap_values.joblib, best_thr.joblib, ohe.joblib)
    ../data/processed/  (GAT_df.csv)
    ../reports/figures/ (créé si absent)
"""
from __future__ import annotations
import os
import gc
import joblib
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from pyrosm import OSM, get_data
from shapely.geometry import box
from branca.element import Template, MacroElement
from matplotlib import cm
from matplotlib.colors import to_hex
import torch

# Constants
REGIONS = [
    "alsace", "aquitaine", "auvergne", "basse-normandie",
    "bourgogne", "bretagne", "centre", "champagne-ardenne",
    "corse", "franche-comte", "haute-normandie", "ile-de-france",
    "languedoc-roussillon", "limousin", "lorraine", "midi-pyrenees",
    "nord-pas-de-calais", "pays-de-la-loire", "picardie",
    "poitou-charentes", "provence-alpes-cote-d-azur", "rhone-alpes"
]

ROAD_OK = {
    "motorway", "trunk", "primary", "secondary", "tertiary",
    "residential", "unclassified", "living_street",
}

# Create output directory
os.makedirs("../reports/figures", exist_ok=True)

# Patch for NumPy >= 1.24
if not hasattr(np, "float"):
    np.float = float

class GATResNet(nn.Module):
    def __init__(self, in_dim, hidden=32, out_dim=2, heads=8, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.gats = nn.ModuleList([
            GATConv(in_dim, hidden, heads=heads, dropout=dropout),
            GATConv(hidden * heads, hidden, heads=heads, dropout=dropout),
            GATConv(hidden * heads, hidden, heads=heads, dropout=dropout),
        ])
        self.bns = nn.ModuleList([
            BatchNorm(hidden * heads),
            BatchNorm(hidden * heads),
            BatchNorm(hidden * heads),
        ])
        self.res0 = nn.Linear(in_dim, hidden * heads, bias=False)
        self.final = GATConv(hidden * heads, out_dim, heads=1, concat=False)

    def forward(self, data):
        x, ei = data.x, data.edge_index
        h = self.gats[0](x, ei)
        h = F.elu(self.bns[0](h))
        x = self.res0(x) + h
        x = F.dropout(x, p=self.dropout, training=self.training)
        for gat, bn in zip(self.gats[1:], self.bns[1:]):
            h = F.elu(bn(gat(x, ei)))
            x = x + h
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.final(x, ei)


def load_model_and_points() -> tuple[gpd.GeoDataFrame, float]:
    print("▶ Chargement modèle et données …")
    model = joblib.load("../models/GAT.joblib")
    data = torch.load("../models/graph_data.pt", map_location="cpu")
    df = pd.read_csv("../data/processed/GAT_df.csv")
    shap_vals = joblib.load("../models/shap_values.joblib")
    best_thr = float(joblib.load("../models/best_thr.joblib"))
    ohe = joblib.load("../models/ohe.joblib")

    num_cols = [
        "vma", "TMJA", "lat_norm", "lon_norm", "nb_accidents",
        "dist_hop_m", "dist_fire_m",
    ]

    cat_cols = [
        "lit", "dep", "agg", "catr", "obs", "obsm", "situ", "prof",
        "plan", "infra", "circ", "int", "bridge", "tunnel", "traffic_calming",
    ]

    feature_names = num_cols + list(ohe.get_feature_names_out(cat_cols))

    model.eval()
    with torch.no_grad():
        logits = model(data).softmax(1).cpu()

    idx_test = data.test_mask.nonzero(as_tuple=False).view(-1)
    probas = logits[:, 1]
    preds = logits.argmax(1)
    sel = (preds[idx_test] == 1) & (probas[idx_test] > best_thr)
    chosen_idx = idx_test[sel]
    print(f"   {len(chosen_idx):,} points à risque retenus > seuil {best_thr:.2f}")

    df_sel = df.loc[chosen_idx.tolist(), ["lat10", "lon10"]].copy()
    df_sel.loc[:, "proba"] = probas[chosen_idx].tolist()
    dec = 4
    df_sel.loc[:, "lat"] = df_sel["lat10"] / 10**dec
    df_sel.loc[:, "lon"] = df_sel["lon10"] / 10**dec
    df_sel.loc[:, "risk"] = 1

    def top3_shap(node_idx: int) -> str:
        vals = shap_vals[node_idx]
        idxs = np.argsort(-np.abs(vals))[:3]
        txts = []
        for i in idxs:
            if i < len(feature_names):
                txts.append(f"{feature_names[i]} {vals[i]:+.2f}")
            else:
                txts.append(f"[feat {i}] {vals[i]:+.2f}")
        return " ; ".join(txts)

    df_sel.loc[:, "shap_top"] = [top3_shap(i) for i in chosen_idx.tolist()]

    gdf_pts = gpd.GeoDataFrame(
        df_sel,
        geometry=gpd.points_from_xy(df_sel.lon, df_sel.lat),
        crs="EPSG:4326",
    )
    return gdf_pts, best_thr

def nearest_join_and_smooth(
    gdf_ways: gpd.GeoDataFrame,
    gdf_pts_reg: gpd.GeoDataFrame,
    best_thr: float,
    max_join_dist: float = 20,
    smooth_dist: float = 200,
) -> gpd.GeoDataFrame:
    join = gpd.sjoin_nearest(
        gdf_ways,
        gdf_pts_reg[["risk", "proba", "shap_top", "geometry"]],
        how="left",
        distance_col="d",
        max_distance=max_join_dist / 111_000,
    )
    for col in ["risk", "proba", "shap_top"]:
        if f"{col}_right" in join.columns and col not in join.columns:
            join = join.rename(columns={f"{col}_right": col})

    join.loc[:, "risk"] = join["risk"].fillna(0).astype(int)
    join.loc[:, "proba"] = join["proba"].fillna(0)
    join.loc[:, "shap_top"] = join["shap_top"].fillna("—")

    gdf_risk = join.query("risk == 1")[["proba", "geometry"]]
    if len(gdf_risk) >= 10:
        gdf_risk = gdf_risk.reset_index(drop=False)
        neigh = (
            gpd.sjoin_nearest(
                gdf_risk, gdf_risk,
                how="left", distance_col="dist",
                max_distance=smooth_dist / 111_000, rsuffix="n",
            )
            .sort_values(["index_left", "dist"])
            .groupby("index_left").head(3)
            .groupby("index_left")["proba_n"].mean()
        )
        join.loc[neigh.index, "proba"] = neigh
        join.loc[neigh.index, "risk"] = 1

    return join[["geometry", "risk", "proba", "shap_top"]]

def download_region_data(region: str) -> str:
    """Download region data if not already present."""
    data_dir = "../data/regions"
    os.makedirs(data_dir, exist_ok=True)
    pbf_path = os.path.join(data_dir, f"{region}-latest.osm.pbf")

    if not os.path.exists(pbf_path):
        print(f"Téléchargement des données pour la région {region}...")
        pbf_path = get_data(region, directory=data_dir)

    return pbf_path

def main() -> None:
    gdf_pts, best_thr = load_model_and_points()
    layers = []

    for reg in REGIONS:
        print(f"▶ Région {reg} …")
        pbf_path = download_region_data(reg)

        gdf_ways = (
            OSM(pbf_path)
            .get_network("driving")
            .to_crs("EPSG:4326")
            .query("highway in @ROAD_OK")
        )

        gdf_ways = gdf_ways[gdf_ways.geometry.is_valid & ~gdf_ways.geometry.is_empty]
        gdf_ways.loc[:, "geometry"] = gdf_ways.geometry.simplify(0.00005)
        gdf_ways = gdf_ways[["geometry"]]

        xmin, ymin, xmax, ymax = gdf_ways.total_bounds
        gdf_pts_reg = gdf_pts.cx[xmin:xmax, ymin:ymax]

        if gdf_pts_reg.empty:
            print("   — aucun point dans cette région —")
            del gdf_ways
            continue

        gdf_join = nearest_join_and_smooth(gdf_ways, gdf_pts_reg, best_thr)
        gdf_join = gdf_join.query("risk == 1").reset_index(drop=True)
        layers.append(gdf_join)

        del gdf_ways, gdf_pts_reg, gdf_join
        gc.collect()

    print("✅ Fusion des régions …")
    gdf_all = gpd.GeoDataFrame(pd.concat(layers, ignore_index=True), crs="EPSG:4326")

    delta = max(1e-6, 1.0 - best_thr)
    reds = [to_hex(c) for c in cm.Reds(np.linspace(0.4, 1, 9))]
    labels = [f"{best_thr + i * delta / 8:.2f}" for i in range(9)]

    def style_fun(feat):
        p = feat["properties"]["proba"]
        idx = int(min(max((p - best_thr) / delta, 0), 1) * 8)
        return {"color": reds[idx], "weight": 2, "opacity": 0.8}

    tooltip = folium.features.GeoJsonTooltip(
        fields=["proba", "shap_top"],
        aliases=["Proba :", "Top SHAP :"],
        localize=True,
    )

    print("✅ Construction carte Folium …")
    m = folium.Map(location=[46.6, 2.2], zoom_start=6, tiles="CartoDB positron")
    g_all_simplified = gdf_all.copy()
    g_all_simplified.loc[:, "geometry"] = g_all_simplified.geometry.simplify(0.00005)
    folium.GeoJson(g_all_simplified, style_function=style_fun, tooltip=tooltip).add_to(m)

    rows = "\n".join(
        f'<rect x="0" y="{i*15}" width="20" height="15" style="fill:{reds[i]};" />'
        f'<text x="25" y="{i*15 + 12}" font-size="12">{labels[i]}</text>'
        for i in range(9)
    )
    legend_html = f"""
    {{% macro html(this, kwargs) %}}
    <div style="
        position: fixed;
        bottom: 50px;
        left: 50px;
        width: 180px;
        z-index:9999;
        font-size:14px;
        background-color: white;
        padding: 10px;
        border:2px solid grey;
        border-radius:5px;
        box-shadow:2px 2px 6px rgba(0,0,0,0.3);
    ">
    <b>Probabilité (gravité)</b><br>
    <svg width="150" height="140">
    {rows}
    </svg>
    </div>
    {{% endmacro %}}
    """
    legend = MacroElement()
    legend._template = Template(legend_html)
    m.get_root().add_child(legend)

    out_path = "../reports/figures/pred_grav_france.html"
    m.save(out_path)
    print(f"🌍 Carte enregistrée : {out_path}")

if __name__ == "__main__":
    main()

▶ Chargement modèle et données …


/var/folders/05/_vxzr9653lg058wd1tp1rdyr0000gn/T/ipykernel_9396/1908947063.py:104: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/GAT_df.csv")


   3,175 points à risque retenus > seuil 0.39
▶ Région alsace …


/opt/anaconda3/envs/env_torch/lib/python3.10/site-packages/pyrosm/networks.py:37: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  edges, nodes = prepare_geodataframe(
/opt/anaconda3/envs/env_torch/lib/python3.10/site-packages/geopandas/array.p

▶ Région aquitaine …


...

▶ Région provence-alpes-cote-d-azur …
Téléchargement des données pour la région provence-alpes-cote-d-azur...
Downloaded Protobuf data 'provence-alpes-cote-d-azur-latest.osm.pbf' (351.32 MB) to:
'/Users/sebastienlebreton/Documents/DataScientest/accidents/mai25_bds_accidents/data/regions/provence-alpes-cote-d-azur-latest.osm.pbf'


/opt/anaconda3/envs/env_torch/lib/python3.10/site-packages/pyrosm/networks.py:37: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  edges, nodes = prepare_geodataframe(
/opt/anaconda3/envs/env_torch/lib/python3.10/site-packages/geopandas/array.p

▶ Région rhone-alpes …
Téléchargement des données pour la région rhone-alpes...
Downloaded Protobuf data 'rhone-alpes-latest.osm.pbf' (470.02 MB) to:
'/Users/sebastienlebreton/Documents/DataScientest/accidents/mai25_bds_accidents/data/regions/rhone-alpes-latest.osm.pbf'


/opt/anaconda3/envs/env_torch/lib/python3.10/site-packages/pyrosm/networks.py:37: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  edges, nodes = prepare_geodataframe(
/opt/anaconda3/envs/env_torch/lib/python3.10/site-packages/geopandas/array.p

✅ Fusion des régions …
✅ Construction carte Folium …
🌍 Carte enregistrée : ../reports/figures/pred_grav_france.html
